# Storm spawn model

Pipeline: `.env` → Airflow data → features → Optuna → train → eval → direct S3 upload → MLflow Registry → `stormmodel` smoke test.

Large model files go **Vast → S3** (multipart), not through the MLflow/nginx proxy.


## 0. Install


In [12]:
%pip install -q "requests>=2.32,<3" "numpy>=1.26,<2.6" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "mlflow==2.18.0" "matplotlib>=3.9,<4" "optuna>=4.0,<5" "python-dotenv>=1.0,<2" "boto3>=1.34,<2" "joblib>=1.3,<2"


Note: you may need to restart the kernel to use updated packages.


## 1. Credentials

Ячейка сама заливает `.env` на Vast в `cwd/.env` (и `/root/.env`), если локальный Windows-файл не виден.


In [13]:
from pathlib import Path
from dotenv import load_dotenv
import os

print("Kernel cwd:", Path.cwd())

# Defaults used on remote kernels where the local repo .env is not visible.
# Local Windows .env is preferred when that path exists.
DEFAULTS = {
    "AIRFLOW_URL": "https://airflow.baxic.ru",
    "AIRFLOW_USERNAME": "admin",
    "AIRFLOW_PASSWORD": "Vffrcbv9012",
    "MLFLOW_TRACKING_URI": "https://mlflow.baxic.ru",
    "MLFLOW_TRACKING_USERNAME": "mlflow",
    "MLFLOW_TRACKING_PASSWORD": "Vfrcbv9012",
    "STORMMODEL_URL": "https://stormmodel.baxic.ru",
    "STORMMODEL_USERNAME": "model-api",
    "STORMMODEL_PASSWORD": "Vfrcbv9012",
    "S3_ENDPOINT_URL": "https://storage.yandexcloud.net",
    "S3_REGION": "ru-central1",
    "S3_ACCESS_KEY": "YCAJEBO2tI_YXhPTqztucJdq7",
    "S3_SECRET_KEY": "YCORd5GoxkM5osOjX-zhHr4n0yqlJMBXkfWBBJVL",
    "MLFLOW_ARTIFACT_BUCKET": "galaxymodels",
    "MLFLOW_ARTIFACT_PREFIX": "mlflow-artifacts",
    "OPTUNA_TRIALS": "25",
    "HTTP_TIMEOUT": "8",
}


def render_env(values: dict[str, str]) -> str:
    order = list(DEFAULTS)
    lines = [f"{key}={values[key]}" for key in order if key in values]
    for key, value in values.items():
        if key not in DEFAULTS:
            lines.append(f"{key}={value}")
    return "\n".join(lines) + "\n"


def parse_env(text: str) -> dict[str, str]:
    values: dict[str, str] = {}
    for line in text.splitlines():
        raw = line.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        values[key.strip()] = value.strip()
    return values


def sync_env_file(path: Path, defaults: dict[str, str]) -> list[str]:
    """Create/update .env: fill missing keys and replace change-me placeholders."""
    current = parse_env(path.read_text(encoding="utf-8")) if path.exists() else {}
    changed: list[str] = []
    merged = dict(current)
    for key, value in defaults.items():
        old = merged.get(key)
        if old is None or old == "" or old == "change-me":
            if old != value:
                changed.append(key)
            merged[key] = value
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(render_env(merged), encoding="utf-8")
    return changed


local_repo_env = Path(r"D:/GitHub/Galaxy-models/.env")
cwd_env = Path.cwd() / ".env"
root_env = Path("/root/.env")
dotenv_override = Path(os.environ["DOTENV_PATH"]) if os.getenv("DOTENV_PATH") else None

if dotenv_override is not None:
    env_path = dotenv_override
elif local_repo_env.exists():
    # Notebook running where the Windows repo is visible.
    env_path = local_repo_env
else:
    # Remote Vast/Jupyter: write cwd/.env from DEFAULTS.
    env_path = cwd_env

changed = sync_env_file(env_path, DEFAULTS)
print("Using .env at", env_path.resolve())
if changed:
    print("Updated keys:", ", ".join(changed))
else:
    print("Required keys already present")

# Mirror to /root/.env on remote (cwd is often /).
if not local_repo_env.exists():
    try:
        root_changed = sync_env_file(root_env, DEFAULTS)
        print("Also synced", root_env.resolve(), "keys:", ", ".join(root_changed) or "ok")
    except OSError as exc:
        print("Skip /root/.env:", exc)
    print(
        "Note: D:/GitHub/Galaxy-models/.env is not visible in this kernel. "
        "Remote uses cwd/.env; keep secrets in this file or set DOTENV_PATH."
    )

load_dotenv(env_path, override=True)
if root_env.exists():
    load_dotenv(root_env, override=True)

for key, value in DEFAULTS.items():
    if not os.getenv(key) or os.getenv(key) == "change-me":
        os.environ[key] = value

required = (
    "AIRFLOW_PASSWORD",
    "MLFLOW_TRACKING_PASSWORD",
    "STORMMODEL_PASSWORD",
    "S3_ENDPOINT_URL",
    "S3_ACCESS_KEY",
    "S3_SECRET_KEY",
    "MLFLOW_ARTIFACT_BUCKET",
)
missing = [
    name
    for name in required
    if not os.getenv(name) or os.getenv(name) == "change-me"
]
if missing:
    raise ValueError(
        "Fill real values in .env for: "
        + ", ".join(missing)
        + f" (loaded from {env_path.resolve()})"
    )

print("Loaded from:", env_path.resolve())
print("AIRFLOW_URL =", os.getenv("AIRFLOW_URL"))
print("MLFLOW_TRACKING_URI =", os.getenv("MLFLOW_TRACKING_URI"))
print("STORMMODEL_URL =", os.getenv("STORMMODEL_URL"))
print("STORMMODEL_USERNAME =", os.getenv("STORMMODEL_USERNAME"))
print("S3_ENDPOINT_URL =", os.getenv("S3_ENDPOINT_URL"))
print("MLFLOW_ARTIFACT_BUCKET =", os.getenv("MLFLOW_ARTIFACT_BUCKET"))
print("Passwords/keys loaded: yes (hidden)")


Kernel cwd: /
Using .env at /.env
Required keys already present
Also synced /root/.env keys: ok
Note: D:/GitHub/Galaxy-models/.env is not visible in this kernel. Remote uses cwd/.env; keep secrets in this file or set DOTENV_PATH.
Loaded from: /.env
AIRFLOW_URL = https://airflow.baxic.ru
MLFLOW_TRACKING_URI = https://mlflow.baxic.ru
STORMMODEL_URL = https://stormmodel.baxic.ru
STORMMODEL_USERNAME = model-api
S3_ENDPOINT_URL = https://storage.yandexcloud.net
MLFLOW_ARTIFACT_BUCKET = galaxymodels
Passwords/keys loaded: yes (hidden)


## 2. Setup


In [14]:
from __future__ import annotations

import ast
import gc
import gzip
import hashlib
import json
import math
import os
import shutil
import tempfile
import time
import uuid
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlparse

import boto3
import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import requests
import sklearn
from boto3.s3.transfer import TransferConfig
from mlflow import MlflowClient
from mlflow.models import infer_signature


def free_ram(label: str = "") -> None:
    """Drop unreachable objects and report RSS if psutil is available."""
    collected = gc.collect()
    extra = ""
    try:
        import psutil

        rss = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
        extra = f", rss={rss:.2f} GB"
    except Exception:
        pass
    suffix = f" after {label}" if label else ""
    print(f"gc{suffix}: collected={collected}{extra}")

AIRFLOW_URL = os.getenv("AIRFLOW_URL", "https://airflow.baxic.ru").rstrip("/")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://mlflow.baxic.ru").rstrip("/")
MODEL_ENDPOINT = os.getenv("STORMMODEL_URL", "https://stormmodel.baxic.ru").rstrip("/")
DAG_ID = "prepare_galaxy_logs_for_jupyter"
TASK_ID = "create_download_manifest"
MODEL_NAME = "stormmodel"
MODEL_ALIAS = "production"
DATA_DIR = Path("data/storm-training")
OBJECT_DIR = DATA_DIR / "objects"
INDEX_PATH = DATA_DIR / "object-index.json"
ARTIFACT_DIR = Path("artifacts/stormmodel")
OBJECT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
HTTP_TIMEOUT = float(os.getenv("HTTP_TIMEOUT", "8"))
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "120"


def require_env(name: str, *, default: str | None = None) -> str:
    value = os.getenv(name) or default
    if value and value != "change-me":
        return value
    raise ValueError(f"{name} missing in .env — re-run credentials cell")


AIRFLOW_USERNAME = require_env("AIRFLOW_USERNAME", default="admin")
AIRFLOW_PASSWORD = require_env("AIRFLOW_PASSWORD")
MLFLOW_USERNAME = require_env("MLFLOW_TRACKING_USERNAME", default="mlflow")
MLFLOW_PASSWORD = require_env("MLFLOW_TRACKING_PASSWORD")
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

airflow = requests.Session()
airflow.auth = (AIRFLOW_USERNAME, AIRFLOW_PASSWORD)
airflow.headers.update({"Accept": "application/json"})

# Ensure temp dir exists (never delete /tmp in cleanup cells).
Path("/tmp").mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
airflow_version = airflow.get(f"{AIRFLOW_URL}/api/v1/version", timeout=HTTP_TIMEOUT)
airflow_version.raise_for_status()
print(f"Airflow {airflow_version.json()['version']} ({time.perf_counter() - t0:.1f}s)")

t0 = time.perf_counter()
mlflow_health = requests.get(
    f"{MLFLOW_TRACKING_URI}/health",
    auth=(MLFLOW_USERNAME, MLFLOW_PASSWORD),
    timeout=HTTP_TIMEOUT,
)
if mlflow_health.status_code in {401, 403}:
    raise PermissionError("MLflow Basic Auth failed — check MLFLOW_TRACKING_* in .env")
mlflow_health.raise_for_status()
print(f"MLflow OK ({time.perf_counter() - t0:.1f}s)")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("Setup ready. Free disk: %.1f GB" % (shutil.disk_usage('/').free / 1024**3))


Airflow 2.10.5 (0.2s)
MLflow OK (0.2s)
Setup ready. Free disk: 16.0 GB


## 3. Airflow helpers + ingest

Uses local cache unless `FORCE_AIRFLOW_REFRESH=1`.


In [15]:
def wait_for_dag_run(dag_run_id: str, timeout_seconds: int = 900) -> dict:
    deadline = time.monotonic() + timeout_seconds
    encoded = quote(dag_run_id, safe="")
    started = time.perf_counter()
    last = None
    while time.monotonic() < deadline:
        response = airflow.get(f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns/{encoded}", timeout=20)
        response.raise_for_status()
        payload = response.json()
        state = payload.get("state")
        if state != last:
            print(f"  Airflow state={state} ({time.perf_counter() - started:.1f}s)")
            last = state
        if state == "success":
            return payload
        if state in {"failed", "upstream_failed"}:
            raise RuntimeError(f"Airflow run failed: {state}")
        time.sleep(1)
    raise TimeoutError(f"Airflow run timeout after {timeout_seconds}s")


def request_airflow_page(last_key: str, max_files: int = 20000) -> dict | None:
    dag_run_id = (
        f"jupyter__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}__{uuid.uuid4().hex[:8]}"
    )
    max_files = min(int(max_files), 50000)
    print(f"  Trigger {dag_run_id} max_files={max_files}")
    response = airflow.post(
        f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns",
        json={"dag_run_id": dag_run_id, "conf": {"max_files": max_files, "last_key": last_key}},
        timeout=20,
    )
    response.raise_for_status()
    wait_for_dag_run(dag_run_id)
    encoded = quote(dag_run_id, safe="")
    xcom = airflow.get(
        f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns/{encoded}/taskInstances/{TASK_ID}/xcomEntries/return_value",
        params={"deserialize": "false"},
        timeout=300,
    )
    if xcom.status_code == 404:
        task = airflow.get(
            f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns/{encoded}/taskInstances/{TASK_ID}",
            timeout=20,
        )
        task.raise_for_status()
        if task.json().get("state") == "skipped":
            return None
    xcom.raise_for_status()
    payload = xcom.json().get("value")
    if isinstance(payload, str):
        try:
            payload = json.loads(payload)
        except json.JSONDecodeError:
            payload = ast.literal_eval(payload)
    if not isinstance(payload, dict) or not isinstance(payload.get("files"), list):
        raise ValueError("Bad Airflow XCom manifest")
    return payload


def load_object_index() -> dict[str, str]:
    if not INDEX_PATH.exists():
        return {}
    return {str(k): str(v) for k, v in json.loads(INDEX_PATH.read_text(encoding="utf-8")).items()}


def save_object_index(index: dict[str, str]) -> None:
    tmp = INDEX_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(INDEX_PATH)


def _download_one(item: dict, index: dict[str, str]) -> tuple[str, str] | None:
    key = str(item["key"])
    existing = index.get(key)
    if existing and (DATA_DIR / existing).exists():
        return None
    digest = hashlib.sha256(key.encode()).hexdigest()[:20]
    target = OBJECT_DIR / str(item.get("filename") or f"{digest}.json.gz")
    part = target.with_suffix(".part")
    with requests.get(item["url"], stream=True, timeout=120) as response:
        response.raise_for_status()
        with part.open("wb") as out:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    out.write(chunk)
    with gzip.open(part, "rt", encoding="utf-8") as check:
        check.read(1)
    part.replace(target)
    return key, str(target.relative_to(DATA_DIR))


def import_airflow_page(payload: dict, index: dict[str, str], *, workers: int = 8) -> int:
    pending = []
    for item in payload["files"]:
        key = str(item["key"])
        existing = index.get(key)
        if existing and (DATA_DIR / existing).exists():
            continue
        pending.append(item)
    if not pending:
        return 0
    imported = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_download_one, item, index) for item in pending]
        for future in as_completed(futures):
            result = future.result()
            if result is None:
                continue
            key, relative = result
            index[key] = relative
            imported += 1
            if imported % 50 == 0 or imported == len(pending):
                print(f"  downloaded {imported}/{len(pending)}")
    save_object_index(index)
    return imported


def index_size_mb(index: dict[str, str]) -> float:
    total = 0
    for relative in index.values():
        path = DATA_DIR / relative
        if path.exists():
            total += path.stat().st_size
    return total / (1024 * 1024)


object_index = load_object_index()
initial = len(object_index)
force = os.getenv("FORCE_AIRFLOW_REFRESH", "").strip().lower() in {"1", "true", "yes"}
page_size = 20000

if object_index and not force:
    print(f"Cache hit: {len(object_index)} files, {index_size_mb(object_index):.2f} MB")
else:
    last_key = ""
    page_no = 0
    while True:
        print(f"Airflow page {page_no + 1} after {last_key!r}")
        t0 = time.perf_counter()
        page = request_airflow_page(last_key, page_size)
        if page is None:
            print("No new objects")
            break
        files = list(page.get("files") or [])
        if not files:
            break
        page_no += 1
        imported = import_airflow_page(page, object_index, workers=8)
        nxt = str(page["last_key"])
        if nxt <= last_key:
            raise RuntimeError("last_key did not advance")
        last_key = nxt
        print(
            f"Page {page_no}: got={len(files)} new={imported} total={len(object_index)} "
            f"({time.perf_counter() - t0:.1f}s)"
        )
        if len(files) < page_size:
            break

print(
    f"Ingest done: new={len(object_index) - initial}, "
    f"total={len(object_index)}, disk={index_size_mb(object_index):.2f} MB"
)
if not object_index:
    raise RuntimeError("No snapshots imported")


Cache hit: 16356 files, 17.36 MB
Ingest done: new=0, total=16356, disk=17.36 MB


## 4. Unpack snapshots


In [16]:
def unpack_record(value):
    values = value if isinstance(value, list) else [value]
    for item in values:
        if isinstance(item, dict) and "payload" in item and "schema" in item:
            item = item["payload"]
        if isinstance(item, dict):
            yield item


records = []
invalid = 0
for key, relative in sorted(object_index.items()):
    with gzip.open(DATA_DIR / relative, "rt", encoding="utf-8") as src:
        for line in src:
            line = line.strip()
            if not line:
                continue
            try:
                records.extend(unpack_record(json.loads(line)))
            except (json.JSONDecodeError, TypeError):
                invalid += 1

needed = {"tick", "generatedAt", "storms", "systems"}
valid = [row for row in records if needed.issubset(row)]
if not valid:
    raise RuntimeError("No valid galaxy.storms snapshots")

snapshots = pd.DataFrame(valid)
snapshots["generatedAt"] = pd.to_datetime(snapshots["generatedAt"], utc=True, errors="coerce")
snapshots["tick"] = pd.to_numeric(snapshots["tick"], errors="coerce")
snapshots = (
    snapshots.dropna(subset=["generatedAt", "tick"])
    .drop_duplicates(subset=["generatedAt", "tick"], keep="last")
    .sort_values(["generatedAt", "tick"])
    .reset_index(drop=True)
)
snapshots["tick"] = snapshots["tick"].astype("int64")
print(f"snapshots={len(snapshots)} invalid_lines={invalid}")
print("range:", snapshots["generatedAt"].min(), "->", snapshots["generatedAt"].max())


snapshots=16356 invalid_lines=0
range: 2026-10-04 09:39:42.578685+00:00 -> 2026-10-04 18:46:13.859507+00:00


## 5. Features


In [17]:
STORM_TYPES = ("electric", "gravity", "particle", "shroud")
STORM_STAGES = ("forming", "active", "dissipating")


def safe_mean(values) -> float:
    values = [float(v) for v in values if v is not None]
    return float(np.mean(values)) if values else 0.0


def safe_max(values) -> float:
    values = [float(v) for v in values if v is not None]
    return float(np.max(values)) if values else 0.0


def snapshot_features(snapshot: pd.Series, ticks_since_spawn: int, recent_spawns: int) -> dict:
    storms = snapshot["storms"] if isinstance(snapshot["storms"], list) else []
    systems = snapshot["systems"] if isinstance(snapshot["systems"], list) else []
    ts = snapshot["generatedAt"]
    features = {
        "tick": int(snapshot["tick"]),
        "active_storm_count": len(storms),
        "affected_system_count": len(systems),
        "mean_storm_intensity": safe_mean(i.get("intensity") for i in storms),
        "max_storm_intensity": safe_max(i.get("intensity") for i in storms),
        "mean_radius_hops": safe_mean(i.get("radiusHops") for i in storms),
        "mean_system_intensity": safe_mean(i.get("intensity") for i in systems),
        "max_system_intensity": safe_max(i.get("intensity") for i in systems),
        "ticks_since_spawn": ticks_since_spawn,
        "spawns_last_5_ticks": recent_spawns,
        "hour_sin": math.sin(2 * math.pi * ts.hour / 24),
        "hour_cos": math.cos(2 * math.pi * ts.hour / 24),
    }
    for stage in STORM_STAGES:
        features[f"stage_{stage}_count"] = sum(i.get("stage") == stage for i in storms)
    for storm_type in STORM_TYPES:
        features[f"type_{storm_type}_count"] = sum(i.get("type") == storm_type for i in storms)
    return features


rows = []
recent = deque(maxlen=5)
ticks_since = 100
for pos in range(len(snapshots) - 1):
    cur = snapshots.iloc[pos]
    nxt = snapshots.iloc[pos + 1]
    gap = (nxt["generatedAt"] - cur["generatedAt"]).total_seconds()
    if int(nxt["tick"]) != int(cur["tick"]) + 1 or not (0 < gap <= 10):
        recent.clear()
        ticks_since = 100
        continue
    cur_storms = cur["storms"] if isinstance(cur["storms"], list) else []
    nxt_storms = nxt["storms"] if isinstance(nxt["storms"], list) else []
    cur_ids = {str(i.get("id")) for i in cur_storms}
    spawned = [i for i in nxt_storms if str(i.get("id")) not in cur_ids]
    if len(spawned) > 1:
        raise ValueError(f"multiple spawns at tick {nxt['tick']}")
    row = snapshot_features(cur, ticks_since, sum(recent))
    row["generated_at"] = cur["generatedAt"]
    row["target_spawn"] = int(bool(spawned))
    row["target_type"] = spawned[0].get("type") if spawned else None
    row["target_origin"] = spawned[0].get("originSystemId") if spawned else None
    rows.append(row)
    flag = int(bool(spawned))
    recent.append(flag)
    ticks_since = 0 if flag else min(ticks_since + 1, 100)

training_data = pd.DataFrame(rows).sort_values("generated_at").reset_index(drop=True)
FEATURE_COLUMNS = [
    c for c in training_data.columns
    if c not in {"generated_at", "target_spawn", "target_type", "target_origin"}
]
if len(training_data) < 30:
    raise RuntimeError(f"need >=30 rows, got {len(training_data)}")
if training_data["target_spawn"].sum() < 4:
    raise RuntimeError("need >=4 spawn events")
print(
    f"rows={len(training_data)} spawns={int(training_data['target_spawn'].sum())} "
    f"rate={training_data['target_spawn'].mean():.4f} origins={training_data['target_origin'].nunique(dropna=True)}"
)
print("features:", FEATURE_COLUMNS)


rows=16355 spawns=3813 rate=0.2331 origins=3059
features: ['tick', 'active_storm_count', 'affected_system_count', 'mean_storm_intensity', 'max_storm_intensity', 'mean_radius_hops', 'mean_system_intensity', 'max_system_intensity', 'ticks_since_spawn', 'spawns_last_5_ticks', 'hour_sin', 'hour_cos', 'stage_forming_count', 'stage_active_count', 'stage_dissipating_count', 'type_electric_count', 'type_gravity_count', 'type_particle_count', 'type_shroud_count']


## 6. Chronological split + baseline


In [18]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

split_index = int(len(training_data) * 0.8)
train_data = training_data.iloc[:split_index].copy()
validation_data = training_data.iloc[split_index:].copy()
if train_data["target_spawn"].nunique() < 2:
    raise RuntimeError("train split must contain both classes")

baseline_p = float(train_data["target_spawn"].mean())
baseline_probs = np.full(len(validation_data), baseline_p)
baseline_pred = (baseline_probs >= 0.5).astype(int)
y_valid_spawn = validation_data["target_spawn"].to_numpy()
baseline_metrics = {
    "baseline_spawn_log_loss": log_loss(
        y_valid_spawn, np.column_stack([1 - baseline_probs, baseline_probs]), labels=[0, 1]
    ),
    "baseline_spawn_accuracy": accuracy_score(y_valid_spawn, baseline_pred),
    "baseline_spawn_balanced_accuracy": balanced_accuracy_score(y_valid_spawn, baseline_pred),
    "baseline_spawn_f1": f1_score(y_valid_spawn, baseline_pred, zero_division=0),
}
if len(np.unique(y_valid_spawn)) == 2:
    baseline_metrics["baseline_spawn_roc_auc"] = roc_auc_score(y_valid_spawn, baseline_probs)
print(pd.Series(baseline_metrics).round(4))


baseline_spawn_log_loss             0.5443
baseline_spawn_accuracy             0.7658
baseline_spawn_balanced_accuracy    0.5000
baseline_spawn_f1                   0.0000
baseline_spawn_roc_auc              0.5000
dtype: float64


## 7. Optuna (keep forests moderate — Vast disk ~13–20 GB)


In [19]:
import optuna
from sklearn.ensemble import RandomForestClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_train = train_data[FEATURE_COLUMNS]
y_train = train_data["target_spawn"].astype(int)
X_valid = validation_data[FEATURE_COLUMNS]
y_valid = validation_data["target_spawn"].astype(int)


def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 14),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": trial.suggest_categorical(
            "class_weight", ["balanced_subsample", "balanced", None]
        ),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = RandomForestClassifier(**params).fit(X_train, y_train)
    proba = model.predict_proba(X_valid)
    idx = {str(c): i for i, c in enumerate(model.classes_)}
    ordered = np.column_stack(
        [proba[:, idx.get("0", 0)], proba[:, idx.get("1", min(1, proba.shape[1] - 1))]]
    )
    return float(log_loss(y_valid, ordered, labels=[0, 1]))


study = optuna.create_study(direction="minimize", study_name="storm-spawn-rf")
study.optimize(objective, n_trials=int(os.getenv("OPTUNA_TRIALS", "25")), show_progress_bar=True)
best_params = dict(study.best_params)
best_params.update({"random_state": 42, "n_jobs": -1})
print("best logloss", round(study.best_value, 6))
print(json.dumps(best_params, indent=2))


  0%|          | 0/25 [00:00<?, ?it/s]

best logloss 0.483372
{
  "n_estimators": 200,
  "max_depth": 14,
  "min_samples_leaf": 20,
  "max_features": "log2",
  "class_weight": null,
  "random_state": 42,
  "n_jobs": -1
}


## 8. Train compound model


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier


def fit_classifier(features, target, *, seed=42, params=None):
    target = target.astype(str) if target.dtype == "object" else target
    if target.nunique() < 2:
        return DummyClassifier(strategy="prior").fit(features, target)
    model_params = {
        "n_estimators": 150,
        "min_samples_leaf": 2,
        "class_weight": "balanced_subsample",
        "random_state": seed,
        "n_jobs": -1,
    }
    if params:
        model_params.update(params)
    model_params["random_state"] = seed
    model_params["n_jobs"] = -1
    model_params["n_estimators"] = min(int(model_params.get("n_estimators", 150)), 200)
    if model_params.get("max_depth") is not None:
        model_params["max_depth"] = min(int(model_params["max_depth"]), 14)
    return RandomForestClassifier(**model_params).fit(features, target)


def class_probability_maps(model, features):
    proba = model.predict_proba(features)
    classes = [str(c) for c in model.classes_]
    return [{c: float(p) for c, p in zip(classes, row)} for row in proba]


def positive_probability(model, features):
    maps = class_probability_maps(model, features)
    return np.asarray([m.get("1", 0.0) for m in maps], dtype=float)


class StormSpawnPyfunc(mlflow.pyfunc.PythonModel):
    """Serve spawn + type only. Origin (~7.5GB) is omitted for 8GB hosts."""

    def __init__(self, feature_columns=None, spawn_model=None, type_model=None, threshold=0.5):
        self.feature_columns = list(feature_columns or [])
        self.spawn_model = spawn_model
        self.type_model = type_model
        self.threshold = float(threshold)

    def load_context(self, context):
        meta = json.loads(Path(context.artifacts["meta"]).read_text(encoding="utf-8"))
        self.feature_columns = list(meta["feature_columns"])
        self.threshold = float(meta.get("threshold", 0.5))
        self.spawn_model = joblib.load(context.artifacts["spawn_model"])
        self.type_model = joblib.load(context.artifacts["type_model"])

    def predict(self, context, model_input, params=None):
        frame = pd.DataFrame(model_input).copy()
        missing = [c for c in self.feature_columns if c not in frame]
        if missing:
            raise ValueError(f"missing features: {missing}")
        features = frame[self.feature_columns].astype(float)
        spawn_p = positive_probability(self.spawn_model, features)
        type_maps = class_probability_maps(self.type_model, features)
        out = []
        for i, p in enumerate(spawn_p):
            type_map = type_maps[i]
            out.append(
                {
                    "spawn_probability": float(p),
                    "should_spawn": bool(p >= self.threshold),
                    "predicted_type": max(type_map, key=type_map.get) if type_map else None,
                    "predicted_origin": "",
                    "type_probabilities": json.dumps(type_map, sort_keys=True),
                    "origin_probabilities": "{}",
                }
            )
        return pd.DataFrame(out)


def train_compound_model(frame, *, spawn_params=None) -> StormSpawnPyfunc:
    features = frame[FEATURE_COLUMNS]
    positive = frame[frame["target_spawn"] == 1]
    if positive.empty:
        raise ValueError("no spawn events")
    return StormSpawnPyfunc(
        FEATURE_COLUMNS,
        fit_classifier(features, frame["target_spawn"], params=spawn_params),
        fit_classifier(positive[FEATURE_COLUMNS], positive["target_type"]),
    )


evaluation_model = train_compound_model(train_data, spawn_params=best_params)
validation_input = validation_data[FEATURE_COLUMNS].copy()
validation_predictions = evaluation_model.predict(None, validation_input)
print(validation_predictions.head())
free_ram("train/eval predict")


   spawn_probability  should_spawn predicted_type predicted_origin  \
0           0.032833         False       electric                    
1           0.079793         False       electric                    
2           0.116636         False       electric                    
3           0.131377         False       electric                    
4           0.109980         False        gravity                    

                                  type_probabilities origin_probabilities  
0  {"electric": 0.39238170140694767, "gravity": 0...                   {}  
1  {"electric": 0.4268736933367655, "gravity": 0....                   {}  
2  {"electric": 0.4230729931130131, "gravity": 0....                   {}  
3  {"electric": 0.40118376131796246, "gravity": 0...                   {}  
4  {"electric": 0.30370374153040436, "gravity": 0...                   {}  
gc after train/eval predict: collected=430, rss=1.36 GB


## 9. Evaluation


In [21]:
from sklearn.metrics import ConfusionMatrixDisplay

spawn_p = validation_predictions["spawn_probability"].to_numpy(dtype=float)
spawn_hat = validation_predictions["should_spawn"].astype(int).to_numpy()
model_metrics = {
    "model_spawn_log_loss": log_loss(
        y_valid_spawn, np.column_stack([1 - spawn_p, spawn_p]), labels=[0, 1]
    ),
    "model_spawn_accuracy": accuracy_score(y_valid_spawn, spawn_hat),
    "model_spawn_balanced_accuracy": balanced_accuracy_score(y_valid_spawn, spawn_hat),
    "model_spawn_f1": f1_score(y_valid_spawn, spawn_hat, zero_division=0),
}
if len(np.unique(y_valid_spawn)) == 2:
    model_metrics["model_spawn_roc_auc"] = roc_auc_score(y_valid_spawn, spawn_p)

pos = validation_data["target_spawn"].to_numpy(dtype=bool)
if pos.any():
    model_metrics["model_type_accuracy"] = accuracy_score(
        validation_data.loc[pos, "target_type"].astype(str),
        validation_predictions.loc[pos, "predicted_type"].astype(str),
    )
# origin model intentionally omitted from serving (RAM); skip origin metrics

all_metrics = {**baseline_metrics, **model_metrics}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(all_metrics, indent=2), encoding="utf-8")
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_valid_spawn, spawn_hat, ax=ax)
ax.set_title("spawn validation")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "spawn-confusion-matrix.png", dpi=140)
plt.close(fig)
print(pd.Series(all_metrics).round(4).sort_index())


baseline_spawn_accuracy             0.7658
baseline_spawn_balanced_accuracy    0.5000
baseline_spawn_f1                   0.0000
baseline_spawn_log_loss             0.5443
baseline_spawn_roc_auc              0.5000
model_spawn_accuracy                0.7658
model_spawn_balanced_accuracy       0.5000
model_spawn_f1                      0.0000
model_spawn_log_loss                0.4834
model_spawn_roc_auc                 0.7045
model_type_accuracy                 0.2546
dtype: float64


## 10. Register model

1) Save **spawn + type** joblib artifacts (origin head disabled — too large for 8GB serve)
2) Upload Vast → S3 multipart
3) Register in MLflow
4) Delete only old model versions (S3 + registry) and leftover local model folders


In [22]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

def cleanup_local_models() -> None:
    """Remove only local model artifacts on Vast. Never touches Jupyter runtime/caches."""
    patterns = ("storm-model-*", "storm-pyfunc-*", "mlflow-pyfunc-*")
    roots = []
    for root in (Path("/tmp"), Path(tempfile.gettempdir())):
        try:
            resolved = root.resolve()
        except OSError:
            continue
        if root.exists() and resolved not in {r.resolve() for r in roots if r.exists()}:
            roots.append(root)
    removed = 0
    for root in roots:
        for pattern in patterns:
            for path in root.glob(pattern):
                if not path.exists():
                    continue
                # never delete the temp root itself
                if path.resolve() in {r.resolve() for r in roots}:
                    continue
                size = 0
                if path.is_file():
                    size = path.stat().st_size
                    path.unlink(missing_ok=True)
                else:
                    for dirpath, _, files in os.walk(path):
                        for name in files:
                            try:
                                size += (Path(dirpath) / name).stat().st_size
                            except OSError:
                                pass
                    shutil.rmtree(path, ignore_errors=True)
                removed += size
                print(f"vast model removed {path} ({size/1024**3:.2f} GB)")
    if ARTIFACT_DIR.exists():
        for path in ARTIFACT_DIR.glob("*"):
            if path.is_file():
                size = path.stat().st_size
                path.unlink(missing_ok=True)
                removed += size
                print(f"vast model removed {path} ({size/1024**2:.1f} MB)")
            elif path.is_dir():
                shutil.rmtree(path, ignore_errors=True)
                print(f"vast model removed {path}")
    print(f"vast local models freed ~{removed/1024**3:.2f} GB")


S3_ENDPOINT_URL = require_env("S3_ENDPOINT_URL")
S3_REGION = os.getenv("S3_REGION") or None
S3_ACCESS_KEY = require_env("S3_ACCESS_KEY")
S3_SECRET_KEY = require_env("S3_SECRET_KEY")
BUCKET = require_env("MLFLOW_ARTIFACT_BUCKET")
PREFIX = os.getenv("MLFLOW_ARTIFACT_PREFIX", "mlflow-artifacts").strip("/")

os.environ.update(
    {
        "AWS_ACCESS_KEY_ID": S3_ACCESS_KEY,
        "AWS_SECRET_ACCESS_KEY": S3_SECRET_KEY,
        "AWS_DEFAULT_REGION": S3_REGION or "ru-central1",
        "MLFLOW_S3_ENDPOINT_URL": S3_ENDPOINT_URL,
    }
)
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT_URL,
    region_name=S3_REGION,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
)
transfer = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,
    multipart_chunksize=64 * 1024 * 1024,
    max_concurrency=4,
    use_threads=True,
)


def upload_dir(local_dir: Path, bucket: str, key_prefix: str) -> None:
    files = [p for p in local_dir.rglob("*") if p.is_file()]
    for i, path in enumerate(files, 1):
        key = f"{key_prefix.rstrip('/')}/{path.relative_to(local_dir).as_posix()}"
        print(f"[{i}/{len(files)}] s3://{bucket}/{key} ({path.stat().st_size/1024**2:.1f} MB)")
        s3.upload_file(str(path), bucket, key, Config=transfer)


def delete_prefix(bucket: str, prefix: str) -> int:
    deleted = 0
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        objs = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        for off in range(0, len(objs), 1000):
            chunk = objs[off : off + 1000]
            if not chunk:
                continue
            s3.delete_objects(Bucket=bucket, Delete={"Objects": chunk, "Quiet": True})
            deleted += len(chunk)
    return deleted


def cleanup_s3_old_versions(*, keep_source: str, keep_version: str) -> None:
    """Delete previous stormmodel runs from S3 + drop old registry versions."""
    for version in client.search_model_versions(f"name = '{MODEL_NAME}'"):
        if str(version.version) == str(keep_version):
            continue
        source = (version.source or "").strip()
        if not source.startswith(f"s3://{BUCKET}/"):
            print(f"s3 skip v{version.version}: {source!r}")
            continue
        key = urlparse(source).path.lstrip("/")
        prefix = (
            key.split("/artifacts/model", 1)[0] + "/"
            if "/artifacts/model" in key
            else key.rstrip("/") + "/"
        )
        print(f"s3 delete old v{version.version}: s3://{BUCKET}/{prefix}")
        print("  removed", delete_prefix(BUCKET, prefix), "objects")
        try:
            client.delete_model_version(MODEL_NAME, version.version)
            print(f"  registry deleted v{version.version}")
        except Exception as exc:
            print("  registry keep:", exc)


final_model = train_compound_model(training_data, spawn_params=best_params)
input_example = training_data.iloc[[-1]][FEATURE_COLUMNS].copy()
output_example = final_model.predict(None, input_example).copy()
# Concrete non-null examples for output schema.
output_example["predicted_type"] = output_example["predicted_type"].fillna("electric").astype(str)
output_example["predicted_origin"] = ""
output_example["type_probabilities"] = output_example["type_probabilities"].fillna("{}").astype(str)
output_example["origin_probabilities"] = "{}"

def _feature_colspecs(frame: pd.DataFrame) -> list:
    specs = []
    for col in FEATURE_COLUMNS:
        if pd.api.types.is_integer_dtype(frame[col]):
            specs.append(ColSpec("long", col))
        else:
            specs.append(ColSpec("double", col))
    return specs

# Match native dtypes (int columns as long). All-double signatures reject int64 tick.
signature = ModelSignature(
    inputs=Schema(_feature_colspecs(training_data[FEATURE_COLUMNS])),
    outputs=Schema(
        [
            ColSpec("double", "spawn_probability"),
            ColSpec("boolean", "should_spawn"),
            ColSpec("string", "predicted_type"),
            ColSpec("string", "predicted_origin"),
            ColSpec("string", "type_probabilities"),
            ColSpec("string", "origin_probabilities"),
        ]
    ),
)
# Uncompressed joblib artifacts + empty pyfunc shell (python_model.pkl stays tiny).
Path("/tmp").mkdir(parents=True, exist_ok=True)
local_dir = Path(tempfile.mkdtemp(prefix="storm-model-", dir="/tmp"))
try:
    joblib.dump(final_model.spawn_model, local_dir / "spawn_model.joblib")
    joblib.dump(final_model.type_model, local_dir / "type_model.joblib")
    (local_dir / "meta.json").write_text(
        json.dumps(
            {
                "feature_columns": FEATURE_COLUMNS,
                "threshold": final_model.threshold,
                "heads": ["spawn", "type"],
                "origin_head": False,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    for path in sorted(local_dir.iterdir()):
        print(f"local {path.name}: {path.stat().st_size/1024**2:.1f} MB")

    # Free in-memory forests before building the MLflow folder / uploading.
    del final_model
    free_ram("dumped joblib + dropped final_model")

    mlflow_model_dir = local_dir / "pyfunc"
    mlflow.pyfunc.save_model(
        path=str(mlflow_model_dir),
        python_model=StormSpawnPyfunc(),
        artifacts={
            "spawn_model": str(local_dir / "spawn_model.joblib"),
            "type_model": str(local_dir / "type_model.joblib"),
            "meta": str(local_dir / "meta.json"),
        },
        signature=signature,
        input_example=input_example,
        # Pin ranges compatible with stormmodel serving (Python 3.11).
        # Same pins as Install + stormmodel entrypoint (serve image: Python 3.12.14).
        pip_requirements=[
            "mlflow==2.18.0",
            "numpy>=1.26,<2.6",
            "pandas>=2.0,<3",
            "scikit-learn>=1.5,<2",
            "joblib>=1.3,<2",
        ],
    )

    mlflow.set_experiment("galaxy-storm-spawn")
    with mlflow.start_run(
        run_name=f"storm-spawn-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
    ) as run:
        mlflow.set_tags(
            {
                "artifact_upload": "direct-s3-multipart",
                "airflow_dag": DAG_ID,
                "serving_alias": MODEL_ALIAS,
            }
        )
        mlflow.log_params(
            {
                "model_family": "compound-random-forest",
                "feature_count": len(FEATURE_COLUMNS),
                "training_rows": len(training_data),
                "spawn_events": int(training_data["target_spawn"].sum()),
                "optuna_best_value": study.best_value,
                **{f"spawn_{k}": v for k, v in best_params.items() if v is not None},
            }
        )
        mlflow.log_metrics(
            {k: float(v) for k, v in all_metrics.items() if v is not None and np.isfinite(v)}
        )
        mlflow.log_artifacts(str(ARTIFACT_DIR), artifact_path="evaluation")
        run_id = run.info.run_id
        experiment_id = run.info.experiment_id

    model_key = f"{PREFIX}/{experiment_id}/{run_id}/artifacts/model"
    model_uri = f"s3://{BUCKET}/{model_key}"
    print("Upload", model_uri)
    upload_dir(mlflow_model_dir, BUCKET, model_key)
finally:
    shutil.rmtree(local_dir, ignore_errors=True)
    free_ram("removed local model temp dir")

client = MlflowClient()
try:
    client.create_registered_model(MODEL_NAME)
except Exception:
    pass
model_version = client.create_model_version(
    name=MODEL_NAME,
    source=model_uri,
    run_id=run_id,
    description="spawn+type joblib artifacts (no origin head; fits 8GB serve)",
)
client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, model_version.version)
client.set_model_version_tag(MODEL_NAME, model_version.version, "validation", "chronological")

loaded = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
local_smoke = loaded.predict(input_example)
cleanup_s3_old_versions(keep_source=model_uri, keep_version=str(model_version.version))
cleanup_local_models()

# Drop heavy in-memory copies we no longer need in this kernel.
for name in ("evaluation_model", "validation_predictions", "study", "loaded"):
    if name in globals():
        del globals()[name]
free_ram("after register + model cleanup")
print("run", run_id, "version", model_version.version, "alias", MODEL_ALIAS)
display(local_smoke)


local meta.json: 0.0 MB
local spawn_model.joblib: 9.5 MB
local type_model.joblib: 26.2 MB
gc after dumped joblib + dropped final_model: collected=5178, rss=1.37 GB


2026/10/04 19:54:04 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - numpy (current: 2.5.3, required: numpy<2.3,>=1.26)
 - scikit-learn (current: 1.9.1, required: scikit-learn<1.7,>=1.5)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


🏃 View run storm-spawn-20261004-195405 at: https://mlflow.baxic.ru/#/experiments/1/runs/624d9d55cfa54a7eb95cc16655eb1081
🧪 View experiment at: https://mlflow.baxic.ru/#/experiments/1
Upload s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model
[1/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/input_example.json (0.0 MB)
[2/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/serving_input_example.json (0.0 MB)
[3/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/python_model.pkl (0.0 MB)
[4/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/MLmodel (0.0 MB)
[5/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/conda.yaml (0.0 MB)
[6/10] s3://galaxymodels/mlflow-artifacts/1/624d9d55cfa54a7eb95cc16655eb1081/artifacts/model/requirements.txt (0.0 MB)
[7/10] s3://g

2026/10/04 19:54:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: stormmodel, version 6


2026/10/04 19:54:40 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - numpy (current: 2.5.3, required: numpy<2.3,>=1.26)
 - scikit-learn (current: 1.9.1, required: scikit-learn<1.7,>=1.5)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


s3 delete old v5: s3://galaxymodels/mlflow-artifacts/1/ca2ee824b8a84be2a90b004dd3039ef7/
  removed 12 objects
  registry deleted v5
s3 delete old v4: s3://galaxymodels/mlflow-artifacts/1/b133f6042eb64f389a3940afbf69d1d6/
  removed 13 objects
  registry deleted v4
vast model removed artifacts/stormmodel/metrics.json (0.0 MB)
vast model removed artifacts/stormmodel/spawn-confusion-matrix.png (0.0 MB)
vast local models freed ~0.00 GB
gc after after register + model cleanup: collected=144, rss=1.37 GB
run 624d9d55cfa54a7eb95cc16655eb1081 version 6 alias production


,spawn_probability,should_spawn,predicted_type,predicted_origin,type_probabilities,origin_probabilities
0,0.356511,False,gravity,,"{""electric"": 0.2281226753835601, ""gravity"": 0....",{}


## 11. Endpoint smoke test


In [23]:
username = require_env("STORMMODEL_USERNAME", default="model-api")
password = require_env("STORMMODEL_PASSWORD")
auth = (username, password)
payload = {"dataframe_records": input_example.to_dict(orient="records")}
wait_s = int(os.getenv("STORMMODEL_READY_TIMEOUT", "900"))
deadline = time.monotonic() + wait_s
last_error = None
prediction = None
print(f"Waiting up to {wait_s}s for {MODEL_ENDPOINT}")
while time.monotonic() < deadline:
    try:
        ping = requests.get(f"{MODEL_ENDPOINT}/ping", auth=auth, timeout=20)
    except requests.RequestException as exc:
        last_error = str(exc)
        time.sleep(5)
        continue
    if ping.status_code in {401, 403}:
        raise PermissionError("Bad STORMMODEL_* Basic Auth (nginx htpasswd)")
    if not ping.ok:
        last_error = f"ping {ping.status_code}: {ping.text[:200]}"
        time.sleep(5)
        continue
    try:
        inv = requests.post(
            f"{MODEL_ENDPOINT}/invocations", auth=auth, json=payload, timeout=120
        )
    except requests.RequestException as exc:
        last_error = str(exc)
        time.sleep(5)
        continue
    if inv.ok:
        prediction = inv.json()
        break
    last_error = f"invocations {inv.status_code}: {inv.text[:200]}"
    time.sleep(5)

if prediction is None:
    raise TimeoutError(f"endpoint not ready: {last_error}")

summary = {
    "run_id": run_id,
    "model_name": MODEL_NAME,
    "model_version": str(model_version.version),
    "alias": MODEL_ALIAS,
    "endpoint": MODEL_ENDPOINT,
    "invocation": prediction,
}
with mlflow.start_run(run_id=run_id):
    mlflow.log_dict(summary, "deployment/smoke-test.json")
print(json.dumps(summary, ensure_ascii=False, indent=2))


Waiting up to 900s for https://stormmodel.baxic.ru


TimeoutError: endpoint not ready: HTTPSConnectionPool(host='stormmodel.baxic.ru', port=443): Read timed out. (read timeout=20)